In [2]:
%pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [1]:
# 필수 라이브러리 설치
%pip install -q diffusers transformers accelerate peft bitsandbytes
%pip install xformers==0.0.27.post2


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 69.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.3/797.3 MB 91.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 102.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 104.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 107.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 106.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 108.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
%pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 98.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [datasets]/15 [datasets]ess]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from diffusers import StableDiffusionXLPipeline

# 모델 로드
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    dtype=torch.float16,
    # use_safetensors=True,
    variant="fp16"
)
pipe.to("cuda")

# 메모리 최적화
pipe.enable_xformers_memory_efficient_attention()

# 이미지 생성
prompt = "A majestic lion in a sunset savanna, highly detailed, 4k"
negative_prompt = "blurry, low quality, distorted"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    width=1024,
    height=1024
).images[0]

image.save("sdxl_output.png")

/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
Keyword arguments {'dtype': torch.float16} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [1]:
# LoRA 파인튜닝(Diffusers + PEFT(LoRA)
# Stable Diffusion XL  모델의 일부 파라메터만 저비용으로 미세조정 하는 설정 
# 대규모 가중치는 고정
# Attention 계층에만 LoRA 어뎁터를 삽입
# 스타일, 캐릭터, 특정 도메인(의료,산업 이미지)특화된 학습 가능
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from peft import LoraConfig, get_peft_model
import torch

# LoRA 설정
lora_config = LoraConfig(
    r = 8,   # lora rank ( 16~32  스타일/캐릭터 학습에 유리 vram 증가)
    lora_alpha=32,   # 스케일링 벡터
    target_modules=[
        'to_q','to_k','to_v', 'to_out.0',  # Attention Layers
        'proj_in','proj_out',   # u-net
        'ff.net.0.proj', 'ff.net.2',  # Feed Forward Network
    ],
    lora_dropout = 0.05,
)
# 학습설정(sample)
training_args = {
    'learning_rate' : 1e-4,
    'train_batch_size' : 1,
    'gradient_accumulation_step' : 4,
    'max_train_steps' : 1000,
    'mixed_precision' : 'fp16'
}

/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


In [2]:
# QLoRA 파인튜닝(메모리 효율적)  -4bit 양자화
from transformers import BitsAndBytesConfig
from diffusers import StableDiffusionXLPipeline
from peft import LoraConfig, prepare_model_for_kbit_training
# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
# QLoRA 설정
qlora_config = LoraConfig(
    r = 4,   # 더 작은 rank 사용 가능
    lora_alpha=16,   # 스케일링 벡터
    target_modules=[
        'to_q','to_k','to_v', 'to_out.0',  # Attention Layers        
    ],
    lora_dropout = 0.05,
)

In [1]:
import torch
from diffusers import StableDiffusionXLPipeline
from peft import LoraConfig, get_peft_model
from peft import get_peft_model

# LoRA파인튜닝
MODEL_NAME = 'stabilityai/stable-diffusion-xl-base-1.0'
OUTPUT_DIR = './sdxl-rora-output'
DATASET_NAME = 'lambdalabs/naruto-blip-captions'

# 학습하이퍼 파라메터
EPOCH=3
BATCH_SIZE=1
LEARNING_RATE=1e-4
GRADIENT_ACCUMULATION_STEPS=4
RESULUTION=1024

# 모델 로드
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    dtype=torch.float16,
    # use_safetensors=True,
    variant="fp16"
)
#Unet에 LoRA적용
unet = pipe.unet
# LoRA 설정
lora_config = LoraConfig(
    r = 8,   # lora rank ( 16~32  스타일/캐릭터 학습에 유리 vram 증가)
    lora_alpha=32,   # 스케일링 벡터
    target_modules=[
        'to_q','to_k','to_v', 'to_out.0',  # Attention Layers
        'proj_in','proj_out',   # u-net
        'ff.net.0.proj', 'ff.net.2',  # Feed Forward Network
    ],
    lora_dropout = 0.05,
)
# LoRA를 모델에 적용
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()  # 학습가능한 파라메터 확인
# gpu로 이동
device = 'cuda' if torch.cuda.is_available() else 'cpu'
unet.to(device)
pipe.to(device)
# pipe.vae.to(device)
# pipe.text_encoder.to(device)
# pipe.text_encoder_2.to(device)
# 메모리 최적화
pipe.enable_xformers_memory_efficient_attention()

/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.11/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.fp16.safeten(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

vae_1_0/diffusion_pytorch_model.fp16.saf(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

Keyword arguments {'dtype': torch.float16} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

trainable params: 21,278,720 || all params: 2,588,742,404 || trainable%: 0.8220


In [6]:
# 데이터셋 로딩 100개만
from datasets import load_dataset
dataset = load_dataset(DATASET_NAME, split='train[:100]')

Repo card metadata block was not found. Setting CardData to empty.


In [13]:
next(iter(dataset))

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1080x1080>,
 'text': 'a man with dark hair and brown eyes'}

In [5]:
# 이미지전처리
from torchvision import transforms
from torch.utils.data import DataLoader
transform = transforms.Compose([
    transforms.Resize((RESULUTION ,RESULUTION)),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])                    
])
def preprocess(example):
    image = example['image'].convert('RGB')
    example['pixel_values'] = transform(image)
    example['caption']=example['text']
    return example
dataset = dataset.map(preprocess, remove_columns=['image'])
dataset.set_format(type='torch',columns=['pixel_values','caption'])
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
# 학습 설정
from diffusers import DDPMScheduler
optimizer = torch.optim.AdamW(unet.parameters(),lr=LEARNING_RATE)
noise_scheduler = DDPMScheduler.from_retrained(MODEL_NAME,subfolder='scheduler')
# 학습루프
unet.train()

for epoch in range(EPOCHS):
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    total_loss = 0
    
    for step, batch in enumerate(progress_bar):
        with torch.cuda.amp.autocast():  # Mixed Precision
            # Latent 인코딩
            latents = pipe.vae.encode(
                batch["pixel_values"].to(device, dtype=torch.float16)
            ).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
            
            # 노이즈 추가
            noise = torch.randn_like(latents)
            timesteps = torch. randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            
            # 텍스트 임베딩
            text_inputs = pipe.tokenizer(
                batch["caption"],
                padding="max_length",
                max_length=77,
                truncation=True,
                return_tensors="pt"
            ).to(device)
            
            encoder_hidden_states = pipe.text_encoder(
                text_inputs. input_ids
            )[0]
            
            # 예측
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states
            ).sample
            
            # Loss
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
        
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer. step()
            optimizer.zero_grad()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} 완료 - 평균 Loss: {avg_loss:.4f}")


Map:   0%|          | 0/1221 [00:00<?, ? examples/s]

KeyboardInterrupt: 